# Tree of Thoughts - 第二部分：搜索策略

## 学习目标
1. 掌握 BFS、DFS、Beam Search 三种策略
2. 理解思维生成和评估
3. 学会选择合适的搜索策略

## 目录
1. [搜索策略概述](#1-搜索策略概述)
2. [BFS 广度优先](#2-bfs-广度优先)
3. [DFS 深度优先](#3-dfs-深度优先)
4. [Beam Search](#4-beam-search)
5. [思维生成与评估](#5-思维生成与评估)
6. [策略选择指南](#6-策略选择指南)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.tree_of_thoughts import (
    ThoughtNode, BFSSearch, DFSSearch, BeamSearch,
    SimpleThoughtGenerator, SimpleThoughtEvaluator
)
print("模块加载成功！")

---
## 1. 搜索策略概述

In [ ]:
print("""
三种搜索策略对比：

┌──────────────┬─────────────┬─────────────┬─────────────┐
│     策略     │    BFS      │    DFS      │ Beam Search │
├──────────────┼─────────────┼─────────────┼─────────────┤
│ 搜索方式     │ 逐层展开    │ 深入优先    │ 保留最优K个 │
│ 内存使用     │ 高          │ 低          │ 中          │
│ 完备性       │ 是          │ 否          │ 否          │
│ 最优性       │ 是(等代价)  │ 否          │ 近似        │
│ 适用场景     │ 浅树        │ 深树        │ 大规模      │
└──────────────┴─────────────┴─────────────┴─────────────┘
""")

---
## 2. BFS 广度优先

### 2.1 原理

In [ ]:
print("""
BFS 搜索过程：

        [根]           第0层
       / | \\
      A  B  C          第1层 ← 先展开这层所有节点
     /|  |  |\\
    D E  F  G H        第2层 ← 再展开这层

特点：
  - 逐层展开
  - 保证找到最短路径
  - 内存消耗大
""")

In [ ]:
# 创建 BFS 搜索
bfs = BFSSearch()

print(f"BFS 配置：")
print("  BFS 搜索已创建")

In [ ]:
# 执行搜索（需要 generator 和 evaluator）
root = ThoughtNode(thought="解决问题", depth=0)
print(f"BFS 搜索准备就绪")
print(f"  根节点: {root.thought}")

---
## 3. DFS 深度优先

### 3.1 原理

In [ ]:
print("""
DFS 搜索过程：

        [根]
       / | \\
      A  B  C
     /|
    D E  ← 先深入到底
   /
  F      ← 到达叶子后回溯

搜索顺序: 根 → A → D → F → 回溯 → E → 回溯 → B → ...

特点：
  - 深入优先
  - 内存消耗小
  - 可能陷入深分支
""")

In [ ]:
# 创建 DFS 搜索
dfs = DFSSearch()

print(f"DFS 配置：")
print("  BFS 搜索已创建")

In [ ]:
# 执行搜索
root = ThoughtNode(thought="深度搜索问题", depth=0)
print(f"DFS 搜索准备就绪")
print(f"  根节点: {root.thought}")

---
## 4. Beam Search

### 4.1 原理

In [ ]:
print("""
Beam Search (beam_width=2)：

        [根]
       / | \\
      A  B  C     ← 生成3个候选
      ↓  ↓        ← 只保留最优2个 (A, B)
     /|  |\\
    D E  F G      ← 展开保留的节点
    ↓    ↓        ← 再保留最优2个
   ...

特点：
  - 平衡探索与利用
  - beam_width 控制宽度
  - 近似最优解
""")

In [ ]:
# 创建 Beam Search
beam = BeamSearch(beam_width=3)

print(f"Beam Search 配置：")
print(f"  束宽: {beam._beam_width}")
print("  BFS 搜索已创建")

In [ ]:
# 执行搜索
root = ThoughtNode(thought="Beam搜索问题", depth=0)
print(f"Beam Search 准备就绪")
print(f"  根节点: {root.thought}")

---
## 5. 思维生成与评估

### 5.1 思维生成器

In [ ]:
# 使用简单生成器
generator = SimpleThoughtGenerator()

# 生成候选思维
parent = ThoughtNode(thought="如何提高代码质量？", depth=0)
candidates = generator.generate(parent, problem="如何提高代码质量？", n_candidates=3)

print(f"生成的候选思维 ({len(candidates)} 个)：")
for i, c in enumerate(candidates, 1):
    print(f"  {i}. {c[:40]}...")

### 5.2 思维评估器

In [ ]:
# 使用简单评估器
evaluator = SimpleThoughtEvaluator()

# 创建测试节点
test_nodes = [
    ThoughtNode(thought="使用代码审查", depth=1),
    ThoughtNode(thought="编写单元测试", depth=1),
    ThoughtNode(thought="使用静态分析工具", depth=1)
]

# 评估节点
for node in test_nodes:
    score = evaluator.evaluate(node, problem="如何提高代码质量？")
    is_solution = evaluator.is_solution(node, problem="如何提高代码质量？")
    print(f"  节点: {node.thought[:30]}...")
    print(f"    分数: {score:.2f}, 是否解: {is_solution}")

---
## 6. 策略选择指南

In [ ]:
print("""
策略选择指南：

选择 BFS 当：
  - 需要最短路径/最少步骤
  - 树比较浅
  - 内存充足

选择 DFS 当：
  - 解在深层
  - 内存有限
  - 不需要最优解

选择 Beam Search 当：
  - 搜索空间大
  - 需要近似最优
  - 平衡效率和质量

beam_width 选择：
  - 小(2-3): 快速但可能错过好解
  - 中(5-10): 平衡
  - 大(>10): 更全面但更慢
""")

---
## 下一步

继续学习 **03c_ToT_Complete.ipynb** 了解完整 ToT 实现